# Stock Trading Agent — Evaluation with LangSmith

This notebook builds a **stock trading assistant** with three tools and evaluates it
using LangSmith datasets and custom scoring.

| Section | What you will learn |
|---------|--------------------|
| **Tools** | `get_price`, `get_news` (Tavily), `buy_ticker` (mock) |
| **Agent** | ReAct agent with `MemorySaver` checkpointer for multi-turn conversations |
| **Evaluation** | Create a LangSmith dataset, define custom evaluators, and run scoring |

## Setup

1. Run the setup cell below first — all other cells depend on it.
2. Set `MODEL_PROVIDER = "openai"` or `"gemini"`.
3. Required keys in `.env`: `LANGCHAIN_API_KEY`, `TAVILY_API_KEY`, plus the provider key.
4. If packages are missing:
   ```bash
   uv add langsmith yfinance tavily-python langgraph
   ```

In [1]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langsmith import Client
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert LANGCHAIN_API_KEY, "LANGCHAIN_API_KEY not found — add it to your .env file"
assert TAVILY_API_KEY, "TAVILY_API_KEY not found — add it to your .env file"

# ── Enable LangSmith tracing ───────────────────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "stock-trading-eval"

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

ls_client = Client()
logger.info("Setup complete — LLM provider: %s", MODEL_PROVIDER)

INFO | Setup complete — LLM provider: openai


---
## Tools

| Tool | Description |
|------|-------------|
| `get_price` | Fetches historical stock prices via Yahoo Finance for a given ticker and date range |
| `get_news` | Searches for related news using Tavily (returns top 3 results) |
| `buy_ticker` | Mock buy-order tool — returns a confirmation string |

In [2]:
@tool
def get_price(ticker: str, date_range: str) -> str:
    """Get stock price for a ticker over a date range.

    Args:
        ticker: Stock ticker symbol (e.g. AAPL, GOOGL, TSLA).
        date_range: Yahoo Finance period string — e.g. '1d', '5d', '1mo', '3mo', '1y'.
    Returns:
        Price summary including latest close, high, low, and period change.
    """
    logger.info("get_price called | ticker=%s  date_range=%s", ticker, date_range)
    stock = yf.Ticker(ticker)
    hist = stock.history(period=date_range)
    if hist.empty:
        return f"Could not fetch price data for {ticker} over {date_range}."

    latest_close = hist["Close"].iloc[-1]
    period_high = hist["High"].max()
    period_low = hist["Low"].min()
    first_close = hist["Close"].iloc[0]
    change = latest_close - first_close
    pct = (change / first_close) * 100

    result = (
        f"{ticker} ({date_range}): latest close ${latest_close:.2f}, "
        f"high ${period_high:.2f}, low ${period_low:.2f}, "
        f"change {change:+.2f} ({pct:+.2f}%)"
    )
    logger.info("get_price result: %s", result)
    return result


logger.info("Tool ready: get_price")

INFO | Tool ready: get_price


In [3]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def get_news(ticker: str, search_term: str) -> str:
    """Search for recent news related to a stock ticker.

    Args:
        ticker: Stock ticker symbol (e.g. AAPL, NVDA).
        search_term: Additional keywords to narrow the search.
    Returns:
        Top 3 news results with title, snippet, and URL.
    """
    query = f"{ticker} {search_term}"
    logger.info("get_news called | query='%s'", query)

    response = tavily_client.search(query=query, max_results=3)
    results = response.get("results", [])

    if not results:
        return f"No news found for '{query}'."

    lines = []
    for i, r in enumerate(results, 1):
        title = r.get("title", "No title")
        snippet = r.get("content", "")[:200]
        url = r.get("url", "")
        lines.append(f"{i}. {title}\n   {snippet}\n   {url}")

    result = "\n\n".join(lines)
    logger.info("get_news returned %d results", len(results))
    return result


logger.info("Tool ready: get_news")

INFO | Tool ready: get_news


In [ ]:
@tool
def buy_ticker(ticker: str, amount: float, target_price: float) -> str:
    """Place buy order for a stock.

    Args:
        ticker: Stock ticker symbol to buy.
        amount: Number of shares to buy.
        target_price: Target price per share for the limit order.
    Returns:
        Confirmation message for the buy order.
    """
    logger.info(
        "buy_ticker called | ticker=%s  amount=%.2f  target_price=%.2f",
        ticker, amount, target_price,
    )
    result = (
        f"Done, a buy order has been set for {ticker} "
        f"with amount of {amount} shares at price ${target_price:.2f}"
    )
    logger.info("buy_ticker result: %s", result)
    return result


logger.info("Tool ready: buy_ticker")

INFO | Tool ready: buy_ticker


---
## Build the Agent

We use `create_react_agent` from LangGraph with a `MemorySaver` checkpointer
so the agent can maintain conversation context across turns.

Each conversation is isolated by `thread_id` in the config.

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = (
    "You are a stock trading assistant. You can look up stock prices, "
    "search for related news, and place mock buy orders. "
    "Always use the tools when the user asks about prices, news, or wants to buy. "
    "Be concise and include specific numbers in your answers."
)

tools = [get_price, get_news, buy_ticker]
checkpointer = MemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

logger.info("Agent created with tools: %s", [t.name for t in tools])

INFO | Agent created with tools: ['get_price', 'get_news', 'buy_ticker']


---
## Test the Agent

Run a few queries to verify each tool works correctly.
We use the same `thread_id` across turns so the agent remembers context.

In [6]:
config = {"configurable": {"thread_id": "test-session-1"}}

# ── Test 1: Price lookup ───────────────────────────────────────────────────────
result = agent.invoke(
    {"messages": [HumanMessage(content="What is the price of NVDA over the last 5 days?")]},
    config=config,
)
print("=== Price Lookup ===")
print(result["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_price called | ticker=NVDA  date_range=5d
INFO | get_price result: NVDA (5d): latest close $175.75, high $177.37, low $164.27, change +4.51 (+2.63%)
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== Price Lookup ===
Over the last 5 days, NVIDIA (NVDA) has the following price summary:
- Latest Close: $175.75
- High: $177.37
- Low: $164.27
- Change: +$4.51 (+2.63%)


In [7]:
# ── Test 2: News search ────────────────────────────────────────────────────────
result2 = agent.invoke(
    {"messages": [HumanMessage(content="Find me news about TSLA related to earnings")]},
    config=config,
)
print("=== News Search ===")
print(result2["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_news called | query='TSLA earnings'
INFO | get_news returned 3 results
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== News Search ===
Here are some recent news articles related to Tesla (TSLA) and earnings:

1. **Tesla, Inc. (TSLA) Valuation Measures & Financial Statistics**  
   Tesla Inc's financial performance has shown significant challenges in recent quarters, particularly in profitability and growth metrics.  
   [Read more](https://finance.yahoo.com/quote/TSLA/key-statistics/)

2. **Tesla Earnings Dates | Upcoming, Forecasted, & Historical**  
   TSLA's next earnings date is unconfirmed for Tuesday, April 21, 2026, after market close.  
   [Read more](https://www.wallstreethorizon.com/tesla-earnings-calendar)

3. **Tesla, Inc. (TSLA) Earnings Report Date - Nasdaq**  
   Data is currently not available regarding the earnings report date.  
   [Read more](https://www.nasdaq.com/market-activity/stocks/tsla/earnings)


In [8]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Should I buy it")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== Buy Order ===
To provide a more informed recommendation, I can check the current price of TSLA and any recent news that might influence your decision. Would you like me to do that?


In [9]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Yes")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_price called | ticker=TSLA  date_range=1d
INFO | get_news called | query='TSLA latest'
INFO | get_price result: TSLA (1d): latest close $381.26, high $383.14, low $374.08, change +0.00 (+0.00%)
INFO | get_news returned 3 results
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== Buy Order ===
Here are the current details for Tesla (TSLA):

- **Latest Close Price**: $381.26
- **High**: $383.14
- **Low**: $374.08
- **Change**: $0.00 (+0.00%)

### Recent News:
1. **TSLA Stock Quote Price and Forecast - CNN**  
   Latest TSLA news includes various market updates. [Read more](https://www.cnn.com/markets/stocks/TSLA)

2. **Stock Price & Latest News - Reuters**  
   Get real-time stock quotes and financial information from Reuters. [Read more](https://www.reuters.com/markets/companies/TSLA.O/)

3. **Tesla Stock Price Quote & News | Robinhood**  
   The current Tesla stock price is $376.69, with a market capitalization of $1.43 trillion and a P/E ratio of 345.69. [Read more](https://robinhood.com/us/en/stocks/TSLA/)

### Recommendation:
- **Current Price**: $381.26
- **Market Sentiment**: The stock is stable with no recent significant changes.
- **Valuation**: The high P/E ratio suggests that the stock may be overvalued.

Consider your investment strategy, risk to

In [25]:
# ── Test 3: Buy order (uses context from previous turns) ──────────────────────
result3 = agent.invoke(
    {"messages": [HumanMessage(content="Buy 100 shares of it at $370")]},
    config=config,
)
print("=== Buy Order ===")
print(result3["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | buy_ticker called | ticker=NVDA  amount=100.00  target_price=370.00
INFO | buy_ticker result: Done, a buy order has been set for NVDA with amount of 100.0 shares at price $370.00
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== Buy Order ===
Your buy order for 100 shares of NVIDIA (NVDA) at $370.00 has been successfully placed. If you need further assistance or have any questions, feel free to ask!


---
## Evaluation with LangSmith

We will:
1. **Create a dataset** with example queries and expected outputs
2. **Define evaluators** — tool-use check + LLM-as-judge for correctness
3. **Run `evaluate()`** and view results in LangSmith

### 1. Create (or load) the Dataset

In [26]:
DATASET_NAME = "stock-trading-agent-eval"

examples = [
    {
        "inputs": {"question": "What is the price of AAPL over the last month?"},
        "outputs": {
            "expected_tool": "get_price",
            "expected_answer": "The response should contain AAPL price data including close price, high, low, and percentage change over the last month.",
        },
    },
    {
        "inputs": {"question": "Find news about NVDA related to AI chips"},
        "outputs": {
            "expected_tool": "get_news",
            "expected_answer": "The response should contain news articles about NVDA related to AI chips, including titles and links.",
        },
    },
    {
        "inputs": {"question": "Buy 50 shares of TSLA at $250"},
        "outputs": {
            "expected_tool": "buy_ticker",
            "expected_answer": "The response should confirm a buy order for TSLA with 50 shares at $250.",
        },
    },
    {
        "inputs": {"question": "What is GOOGL stock price for the past 5 days?"},
        "outputs": {
            "expected_tool": "get_price",
            "expected_answer": "The response should contain GOOGL price data over 5 days with close, high, low, and change.",
        },
    },
    {
        "inputs": {"question": "Search for recent AAPL news about iPhone sales"},
        "outputs": {
            "expected_tool": "get_news",
            "expected_answer": "The response should contain news articles about AAPL and iPhone sales.",
        },
    },
    {
        "inputs": {"question": "Place a buy order for 100 shares of AAPL at $190"},
        "outputs": {
            "expected_tool": "buy_ticker",
            "expected_answer": "The response should confirm a buy order for AAPL with 100 shares at $190.",
        },
    },
]

# Delete existing dataset if it exists, then create fresh
existing = list(ls_client.list_datasets(dataset_name=DATASET_NAME))
if existing:
    ls_client.delete_dataset(dataset_id=existing[0].id)
    logger.info("Deleted existing dataset: %s", DATASET_NAME)

dataset = ls_client.create_dataset(
    dataset_name=DATASET_NAME,
    description="Evaluation dataset for stock trading agent with price, news, and buy queries.",
)

ls_client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)

logger.info("Dataset '%s' created with %d examples", DATASET_NAME, len(examples))

INFO | Dataset 'stock-trading-agent-eval' created with 6 examples


### 2. Define Evaluators

We define two evaluators:

| Evaluator | What it checks |
|-----------|---------------|
| `correct_tool_used` | Did the agent call the expected tool? (from intermediate messages) |
| `answer_correctness` | LLM-as-judge — does the final answer match the expected criteria? |

In [27]:
def correct_tool_used(outputs: dict, reference_outputs: dict) -> bool:
    """Check whether the agent called the expected tool."""
    expected_tool = reference_outputs.get("expected_tool", "")
    messages = outputs.get("messages", [])

    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                if tc["name"] == expected_tool:
                    logger.debug("Found expected tool call: %s", expected_tool)
                    return True
    logger.debug("Expected tool '%s' was NOT called", expected_tool)
    return False


judge_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)


async def answer_correctness(outputs: dict, reference_outputs: dict) -> bool:
    """LLM-as-judge: does the final answer satisfy the expected criteria?"""
    actual_answer = outputs.get("messages", [{}])[-1].content
    expected = reference_outputs.get("expected_answer", "")

    instructions = (
        "You are an evaluation judge. Given an actual answer from a stock trading agent "
        "and the expected criteria, determine whether the actual answer satisfies the criteria. "
        "Respond ONLY with 'CORRECT' or 'INCORRECT'."
    )
    user_msg = f"ACTUAL ANSWER:\n{actual_answer}\n\nEXPECTED CRITERIA:\n{expected}"

    response = await judge_llm.ainvoke(
        [
            {"role": "system", "content": instructions},
            {"role": "user", "content": user_msg},
        ]
    )
    return response.content.strip().upper() == "CORRECT"


logger.info("Evaluators defined: correct_tool_used, answer_correctness")

INFO | Evaluators defined: correct_tool_used, answer_correctness


### 3. Run Evaluation

We wrap the agent so each evaluation example runs in its own thread (no shared memory).
Then we call `aevaluate()` against the dataset.

In [30]:
from langsmith import aevaluate

eval_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)


async def agent_target(inputs: dict) -> dict:
    """Adapter: convert dataset input format to agent invocation."""
    return await eval_agent.ainvoke(
        {"messages": [{"role": "user", "content": inputs["question"]}]},
    )


experiment_results = await aevaluate(
    agent_target,
    data=DATASET_NAME,
    evaluators=[correct_tool_used, answer_correctness],
    experiment_prefix="stock-agent-eval",
    max_concurrency=2,
)

logger.info("Evaluation complete — check results in LangSmith dashboard")

/Users/danhmac/Documents/agentic_bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'stock-agent-eval-96e81a80' at:
https://smith.langchain.com/o/56508c90-17f3-59d2-8160-7e788ed2739a/datasets/2701a68b-bc8f-4152-993c-3b4d94b941e2/compare?selectedSessions=ae65a02a-7ec6-499e-a938-141b318090d9




0it [00:00, ?it/s]INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_price called | ticker=GOOGL  date_range=5d
INFO | get_price result: GOOGL (5d): latest close $297.39, high $300.52, low $272.11, change +16.47 (+5.86%)
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | buy_ticker called | ticker=AAPL  amount=100.00  target_price=190.00
INFO | buy_ticker result: Done, a buy order has been set for AAPL with amount of 100.0 shares at price $190.00
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
1it [00:05,  5.30s/it]INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_news called | query='AAPL iPhone sales'
INFO | HTTP Request: POST https://api.

In [ ]:
experiment_results.to_pandas()